# Multi-Table Joins

## Overview
Real queries routinely join 4–6 tables. Chaining joins correctly requires understanding join order, cardinality at each step, and how to keep queries readable and maintainable.

**Rules for chaining joins:**
- Each `JOIN` introduces a new table into the running result set
- The ON clause can reference any table already introduced above it
- The query optimiser (especially PostgreSQL) reorders joins for efficiency — you don't need to manually tune join order in most cases
- Readable formatting matters: one join per line, consistent alias conventions

**Readability strategies:**
- Descriptive two-letter aliases (`pt` for patients, `pr` for providers, `en` for encounters)
- Line-break before each JOIN keyword
- ON clause indented under JOIN
- CTEs to break complex multi-joins into named stages

**Cardinality check before aggregating:**
After each join, ask: *could this produce duplicate rows?* A one-to-many join inflates row counts. Always verify with `COUNT(*)` before building aggregates.

---

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE patients (
    patient_id INTEGER PRIMARY KEY, first_name TEXT, last_name TEXT,
    dob TEXT, province TEXT, active INTEGER DEFAULT 1
);
CREATE TABLE providers (
    provider_id INTEGER PRIMARY KEY, full_name TEXT, specialty TEXT,
    province TEXT, supervisor_id INTEGER
);
CREATE TABLE departments (
    dept_id INTEGER PRIMARY KEY, dept_name TEXT, building TEXT, head_provider_id INTEGER
);
CREATE TABLE encounters (
    enc_id INTEGER PRIMARY KEY, patient_id INTEGER, provider_id INTEGER,
    dept_id INTEGER, enc_date TEXT, diag_code TEXT, cost_cad REAL, admitted INTEGER DEFAULT 0
);
CREATE TABLE diagnoses (
    diag_code TEXT PRIMARY KEY, description TEXT, category TEXT
);
CREATE TABLE referrals (
    referral_id INTEGER PRIMARY KEY, from_provider INTEGER, to_provider INTEGER,
    patient_id INTEGER, referral_date TEXT, reason TEXT
);
INSERT INTO patients VALUES
  (1,'Aroha','Ngata','1985-03-12','NB',1),(2,'Liam','Chen','1972-11-04','NS',1),
  (3,'Fatima','Al-Rashid','1990-07-22','ON',1),(4,'James','MacLeod','1955-01-30','NB',0),
  (5,'Sofia','Petrov','2001-09-15','BC',1),(6,'Noah','Williams','1968-06-08','AB',1),
  (7,'Mei','Zhang','1995-04-17','ON',1),(8,'Ethan','Tremblay','1980-12-01','QC',0),
  (9,'Priya','Nair','1977-08-25','BC',1),(10,'Marcus','Okafor','1993-05-19','ON',1),
  (11,'Dana','Leblanc','2000-02-14','NB',1);
INSERT INTO providers VALUES
  (10,'Dr. Sarah MacNeil','Cardiology','NB',NULL),
  (11,'Dr. James Wong','Oncology','BC',10),
  (12,'Dr. Fatima Al-Rashid','Family Medicine','ON',10),
  (13,'Dr. Ethan Tremblay','Orthopaedics','QC',10),
  (14,'Dr. Priya Nair','Emergency Medicine','BC',11),
  (15,'Dr. Marcus Okafor','Radiology','ON',11),
  (16,'Dr. Unassigned','Neurology','NB',12);
INSERT INTO departments VALUES
  (1,'Cardiology','Building A',10),(2,'Oncology','Building B',11),
  (3,'Family Medicine','Building C',12),(4,'Orthopaedics','Building A',13),
  (5,'Emergency','Building D',14),(6,'Radiology','Building B',15),(7,'Neurology','Building C',16);
INSERT INTO diagnoses VALUES
  ('J18.9','Pneumonia, unspecified','Respiratory'),
  ('I25.1','Atherosclerotic heart disease','Cardiovascular'),
  ('Z00.0','General adult examination','Preventive'),
  ('M17.1','Primary osteoarthritis, knee','Musculoskeletal'),
  ('J06.9','Acute upper respiratory infection','Respiratory'),
  ('R07.9','Chest pain, unspecified','Cardiovascular'),
  ('I10','Essential hypertension','Cardiovascular'),
  ('R55','Syncope and collapse','Neurological'),
  ('I48.0','Paroxysmal atrial fibrillation','Cardiovascular'),
  ('S52.5','Fracture of lower end of radius','Musculoskeletal'),
  ('M54.5','Low back pain','Musculoskeletal');
INSERT INTO encounters VALUES
  (1,1,10,1,'2023-01-15','J18.9',1850.00,1),(2,2,10,1,'2023-02-20','I25.1',3200.00,1),
  (3,3,12,3,'2023-03-05','Z00.0',120.00,0),(4,4,13,4,'2023-03-18','M17.1',5500.00,1),
  (5,5,14,5,'2023-04-02','J06.9',95.00,0),(6,6,10,1,'2023-05-11','R07.9',780.00,0),
  (7,7,10,1,'2023-06-22','I10',2100.00,1),(8,8,12,3,'2023-07-14',NULL,80.00,0),
  (9,1,14,5,'2023-08-30','R55',410.00,0),(10,3,12,3,'2023-09-12','Z00.0',110.00,0),
  (11,9,10,1,'2023-10-01','I48.0',1750.00,1),(12,10,13,4,'2023-11-03','S52.5',2200.00,1),
  (13,2,12,3,'2023-11-18',NULL,90.00,0),(14,5,13,4,'2023-12-05','M54.5',450.00,0);
INSERT INTO referrals VALUES
  (1,12,10,3,'2023-03-10','Chest pain follow-up'),
  (2,10,11,2,'2023-03-01','Suspected malignancy'),
  (3,14,10,9,'2023-09-05','AF workup'),
  (4,12,13,5,'2023-12-01','Back pain unresponsive to treatment'),
  (5,10,15,6,'2023-06-15','Imaging for chest pain');
""")
conn.commit()
print("Schema ready — 6 tables")

# Verify base counts
for t in ['patients','providers','departments','encounters','diagnoses','referrals']:
    n = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'  {t}: {n} rows')


---
## Four-table join: full encounter record

In [ ]:
# A complete encounter report: patient + provider + department + diagnosis
print("=== Full encounter record (4 tables) ===")
q = """
SELECT  en.enc_id,
        en.enc_date,
        pt.first_name || ' ' || pt.last_name  AS patient,
        pt.province                            AS patient_province,
        pr.full_name                           AS provider,
        pr.specialty,
        dp.dept_name,
        dp.building,
        di.description                         AS diagnosis,
        di.category                            AS diag_category,
        en.cost_cad,
        CASE en.admitted WHEN 1 THEN 'Admitted' ELSE 'Outpatient' END AS visit_type
FROM    encounters  AS en
INNER JOIN patients    AS pt ON en.patient_id  = pt.patient_id
INNER JOIN providers   AS pr ON en.provider_id = pr.provider_id
INNER JOIN departments AS dp ON en.dept_id     = dp.dept_id
LEFT  JOIN diagnoses   AS di ON en.diag_code   = di.diag_code   -- LEFT: keep uncoded encounters
ORDER BY en.enc_date
"""
df = pd.read_sql(q, conn)
print(df.to_string(index=False))
print(f"\n{len(df)} rows — encounters 8 and 13 appear with NULL diagnosis (uncoded, kept by LEFT JOIN)")


---
## Five-table join: referrals with full context

In [ ]:
# Full referral context: referring provider + receiving provider + patient + departments
print("=== Full referral context (5 tables, providers joined twice) ===")
q = """
SELECT  rf.referral_id,
        rf.referral_date,
        pt.first_name || ' ' || pt.last_name  AS patient,
        src.full_name                          AS referring_provider,
        src_dp.dept_name                       AS referring_dept,
        dst.full_name                          AS receiving_provider,
        dst_dp.dept_name                       AS receiving_dept,
        rf.reason
FROM    referrals   AS rf
INNER JOIN patients    AS pt     ON rf.patient_id    = pt.patient_id
INNER JOIN providers   AS src    ON rf.from_provider = src.provider_id
INNER JOIN providers   AS dst    ON rf.to_provider   = dst.provider_id
INNER JOIN departments AS src_dp ON src_dp.head_provider_id = src.provider_id
INNER JOIN departments AS dst_dp ON dst_dp.head_provider_id = dst.provider_id
ORDER BY rf.referral_date
"""
print(pd.read_sql(q, conn).to_string(index=False))


---
## Using CTEs to break down a complex multi-join

In [ ]:
# CTEs make multi-step joins readable by naming intermediate results
print("=== Provider summary: encounters + referrals sent + referrals received ===")
q = """
WITH enc_summary AS (
    SELECT  provider_id,
            COUNT(*)                     AS total_encounters,
            SUM(admitted)                AS admissions,
            ROUND(SUM(cost_cad), 2)      AS total_billed
    FROM    encounters
    GROUP BY provider_id
),
ref_sent AS (
    SELECT  from_provider AS provider_id,
            COUNT(*)      AS referrals_sent
    FROM    referrals
    GROUP BY from_provider
),
ref_recv AS (
    SELECT  to_provider   AS provider_id,
            COUNT(*)      AS referrals_received
    FROM    referrals
    GROUP BY to_provider
)
SELECT  pr.full_name                              AS provider,
        pr.specialty,
        COALESCE(es.total_encounters, 0)          AS encounters,
        COALESCE(es.admissions, 0)                AS admissions,
        COALESCE(es.total_billed, 0)              AS total_billed,
        COALESCE(rs.referrals_sent, 0)            AS referrals_sent,
        COALESCE(rr.referrals_received, 0)        AS referrals_received
FROM    providers    AS pr
LEFT JOIN enc_summary AS es ON pr.provider_id = es.provider_id
LEFT JOIN ref_sent    AS rs ON pr.provider_id = rs.provider_id
LEFT JOIN ref_recv    AS rr ON pr.provider_id = rr.provider_id
ORDER BY encounters DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))


---
## Checking for row multiplication before aggregating

In [ ]:
# Always verify cardinality after a join before computing aggregates
print("=== Cardinality check: does joining diagnoses inflate encounter rows? ===")

base = conn.execute("SELECT COUNT(*) FROM encounters").fetchone()[0]
after_inner = conn.execute("""
    SELECT COUNT(*) FROM encounters AS e
    INNER JOIN diagnoses AS d ON e.diag_code = d.diag_code
""").fetchone()[0]
after_left = conn.execute("""
    SELECT COUNT(*) FROM encounters AS e
    LEFT JOIN diagnoses AS d ON e.diag_code = d.diag_code
""").fetchone()[0]

print(f"  Base encounters:                {base}")
print(f"  After INNER JOIN diagnoses:     {after_inner}  (coded encounters only)")
print(f"  After LEFT JOIN diagnoses:      {after_left}  (all encounters, NULL for uncoded)")
print()
print("One-to-one join (each diag_code in diagnoses is unique) — no row inflation here.")
print()

# Demonstrate fan-out with a deliberate duplicate in lookup
conn.execute("INSERT INTO diagnoses VALUES ('J18.9','Pneumonia duplicate entry','Respiratory')")
conn.commit()
after_dup = conn.execute("""
    SELECT COUNT(*) FROM encounters AS e
    INNER JOIN diagnoses AS d ON e.diag_code = d.diag_code
""").fetchone()[0]
print(f"  After adding duplicate J18.9 to diagnoses: {after_dup} rows (was {after_inner})")
print("  Fan-out! Encounter 1 (J18.9) now appears TWICE. Always check for duplicates in lookup tables.")

conn.close()


---
## Common Pitfalls

**1. Not checking for row multiplication before aggregating**
Joining a fact table to a dimension with duplicate keys inflates every aggregate silently. Always run `SELECT COUNT(*)` before and after adding a join to verify no fan-out occurred. One duplicate in a lookup table doubles all affected rows.

**2. Mixing INNER and OUTER joins in unexpected order**
`FROM a LEFT JOIN b INNER JOIN c ON c.id = b.id` — the INNER JOIN on c is applied before the LEFT JOIN result is complete, effectively converting the LEFT JOIN into an INNER JOIN for rows where c has no match. Be explicit and test edge cases when mixing join types.

**3. Ambiguous column names across many tables**
With 5+ tables, `ORDER BY province` or `GROUP BY dept_id` is ambiguous if multiple tables have those columns. Use table aliases consistently and prefix every column reference.

**4. CTEs are not always materialised — they may be inlined**
In PostgreSQL, a CTE is an optimisation fence only when written as `WITH ... AS MATERIALIZED (...)`. An ordinary CTE may be inlined by the planner, meaning it could be executed multiple times. For expensive intermediate results that are used more than once, use `MATERIALIZED`.

**5. JOIN order hints are rarely needed but misuse harms performance**
The PostgreSQL query planner handles join order well for up to ~8 tables. Beyond that, join ordering can become suboptimal. Don't manually reorder joins as a performance "fix" without checking `EXPLAIN ANALYZE` first — you may make things worse.

**6. Reading a long join chain without aliases is error-prone**
Reviewing a query with 6 bare table names and column references is difficult. Establish a consistent alias convention (`pt`=patients, `pr`=providers, `en`=encounters, `dp`=departments, `di`=diagnoses, `rf`=referrals) and use it across all notebooks and production queries.


---
*sql_methods_library - Samantha McGarrigle*